In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.patches import Polygon
from scipy.spatial import ConvexHull

from avstack.geometry import q_cam_to_stan, q_mult_vec, transformations as tforms

In [ ]:
agent_colors = list(mcolors.TABLEAU_COLORS.keys())
track_colors = list(mcolors.XKCD_COLORS.keys())


def get_track_color(ID_track):
    return track_colors[ID_track % len(track_colors)]


def get_agent_color(ID_agent):
    return agent_colors[ID_agent % len(agent_colors)]

In [ ]:
cam_extrinsics[0][:3,:3]

In [ ]:
n_cameras = 4

# ----------------------------------------
# EXTRINSICS -- from 3D world to 3D camera
# ----------------------------------------

# rotation
cam_rotations = {
    0: q_cam_to_stan,
    1: q_cam_to_stan,
    2: q_cam_to_stan,
    3: q_cam_to_stan,
}

# position -- specified in world frame, converted to sensor frame
cam_positions = {
    0: np.array([0, 0, 0]),
    1: np.array([0, 5, 0]),
    2: np.array([5, 5, 0]),
    3: np.array([5, 0, 0]),
}

# full transforms
cam_extrinsics = {
    i: np.block(
        [
            [
                tforms.transform_orientation(cam_rotations[i], "quat", "dcm"),
                np.reshape(
                    q_mult_vec(cam_rotations[i].conjugate(), cam_positions[i]), (3, 1)
                ),
            ],
            [np.zeros((1, 3)), np.ones((1, 1))],
        ]
    )
    for i in range(n_cameras)
}

# ----------------------------------------
# INTRINSICS - from 3D camera to 2D image
# ----------------------------------------
P_cam = np.array(
    [
        [300, 0, 1000],
        [0, 300, 1000],
        [0, 0, 1],
    ]
)
cam_intrinsics = {i: P_cam for i in range(n_cameras)}
img_size = {i: (2000, 2000) for i in range(n_cameras)}

In [ ]:
# Create the discretized grid of points in BEV
extent = [[-25, 25], [-25, 25]]
dx = 0.1
X, Y = np.meshgrid(*[np.arange(extent[dim][0], extent[dim][1], dx) for dim in range(2)])
pt_bev_tuples = np.vstack([X.ravel(), Y.ravel(), np.zeros(X.size)])
pt_bev_tuples_homog = np.vstack([pt_bev_tuples, np.ones(pt_bev_tuples.shape[1])])

In [ ]:
# Check the point visibility
boundary = {}
for cam in range(n_cameras):
    # project into camera image plane
    pts_in_cam = cam_extrinsics[cam] @ pt_bev_tuples_homog
    pts_in_img_plane = (cam_intrinsics[cam] @ np.delete(pts_in_cam, 2, axis=0))[:2, :]

    # check in field of view
    visible_this_cam = (
        (-img_size[cam][0] / 2 <= pts_in_img_plane[0, :])
        & (pts_in_img_plane[0, :] <= img_size[cam][0] / 2)
        & (-img_size[cam][1] / 2 <= pts_in_img_plane[1, :])
        & (pts_in_img_plane[1, :] <= img_size[cam][1] / 2)
    )
    if sum(visible_this_cam) == 0:
        raise RuntimeError(f"No points found in FOV for camera {cam}")

    # perform convex hull estimation for boundary
    hull = ConvexHull(pt_bev_tuples[:2, visible_this_cam].T)
    boundary[cam] = [pt_bev_tuples[:2, v] for v in hull.vertices]

In [ ]:
# Plot the result
fig, ax = plt.subplots()
for cam, pos in cam_positions.items():
    # camera positions
    ax.scatter(
        pos[0], pos[1], marker="x", label=f"Camera {cam}", color=get_agent_color(cam)
    )
    # camera fovs
    p = Polygon(boundary[cam], facecolor=get_agent_color(cam), alpha=0.1)
    ax.add_patch(p)
ax.set_xlim(extent[0])
ax.set_ylim(extent[1])
plt.legend()
plt.show()